# Transaction Foundation Model on Ray — Part 4: Pretrain with Ray Train

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~5 min

---

In Part 3 we turned each card's history into token windows. Here we pretrain the foundation model on those windows by masked-feature modeling, using Ray Train to run a plain-PyTorch training loop across the cluster.

In [ ]:
import sys, os, json

DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import pandas as pd

import ray
ray.init(ignore_reinit_error=True, runtime_env={"working_dir": DEMO_ROOT})

from src.paths import artifact_paths, get_demo_base_dir
from src.scale_config import load_scale

SCALE = "mini"                       # same knob as the earlier parts; mini = tiny, CPU-only
cfg = load_scale(SCALE)
BASE = get_demo_base_dir()
paths = artifact_paths(BASE, SCALE)

## Pretrain by predicting masked fields

We pretrain the model the way BERT pretrains on text: mask a fraction (15%) of the dynamic-field tokens across a window and have the model predict the originals from the surrounding transactions. There are no labels involved — the transactions supervise themselves, which is what lets a foundation model learn from far more data than the scarce fraud labels alone.

The encoder is **bidirectional** because the tasks this model feeds (fraud, churn, credit) are fixed-window classification, where seeing both sides of a position beats left-to-right next-token prediction. Each dynamic field gets its own prediction head, and because the heads differ wildly in difficulty — `day_of_week` is 10-way, `merchant` is ~2,000-way — they're balanced with learned **uncertainty weighting** (Kendall & Gal) so the hard head doesn't dominate the gradient. The model and the loss live in `src/model.py`; they're deliberately small, because the model is not the hard part here — the engineering around it is.

## The training loop is plain PyTorch — Ray Train scales it

The function that does the work (`train_func` in `src/pretrain.py`) is ordinary PyTorch: iterate batches, mask, forward, backward, step. Ray Train wraps it with the distributed machinery you'd otherwise hand-write — it launches the workers, shards the dataset across them (`get_dataset_shard` + `iter_torch_batches`), wraps the model in DDP (`prepare_model`), and persists checkpoints to shared storage.

The payoff is the `ScalingConfig`: moving from one CPU worker here to many GPU workers at `full` is a one-line change (`num_workers`, `use_gpu`), not a rewrite of the loop. `scripts/03_pretrain.py` wraps this same `train_func` for headless runs, so the walkthrough and the job train identically.

In [ ]:
from ray.train import ScalingConfig, RunConfig
from ray.train.torch import TorchTrainer
from src.pretrain import train_func, save_checkpoint   # train_func is shared with scripts/03_pretrain.py

pt = cfg["pretrain"]
seq_len = cfg["tokenize"]["seq_len"]
train_ds = ray.data.read_parquet(paths["tokenized_pretrain"])

trainer = TorchTrainer(
    train_func,                                       # the per-worker PyTorch loop (src/pretrain.py)
    train_loop_config={
        "vocab_path": paths["vocab"], "arch": cfg["model"], "size": SCALE, "max_len": seq_len,
        "epochs": pt["epochs"], "batch_size": pt["batch_size"], "lr": pt["lr"],
        "use_fsdp": pt["use_fsdp"], "loss_weighting": "uncertainty", "seed": 0,
    },
    # The one line that moves laptop -> cluster: number of workers, CPU vs GPU.
    scaling_config=ScalingConfig(num_workers=pt["num_workers"], use_gpu=pt["use_gpu"]),
    datasets={"train": train_ds},                     # Ray Train shards this across the workers
    run_config=RunConfig(
        name="transaction_fm_pretrain",
        storage_path=os.path.join(BASE, "ray_results"),  # shared storage: workers run on other nodes
    ),
)
result = trainer.fit()
save_checkpoint(result, paths["checkpoint"])           # copy weights to the canonical path for Part 5
print(f"done — final mlm_loss {result.metrics['mlm_loss']:.2f}, "
      f"macro accuracy {result.metrics['acc_macro']:.3f} -> {paths['checkpoint']}")

## Reading the metrics

Watch the per-field numbers, not the weighted total — the total drifts as the uncertainty weights learn. For each dynamic field we report top-1 accuracy and perplexity. The tell that the model is learning structure rather than guessing is **perplexity well below the field's vocab size**: a head that learned nothing sits around the vocab size.

At `mini` (2 CPU epochs, a 2-layer model) this is only a smoke test, but even here most fields clear that bar — amount lands around perplexity 6 against 19 buckets, `merchant_category` around 7 against 12. The real signal, and the downstream fraud lift, comes from the `small`/`full` GPU pretrain.

In [ ]:
from src.tokenizer import DYNAMIC_FIELDS, field_vocab_sizes

m = result.metrics
sizes = field_vocab_sizes()
print(f"final MLM loss {m['mlm_loss']:.2f}   ·   macro accuracy {m['acc_macro']:.3f}\n")
tbl = pd.DataFrame([
    {"field": f, "accuracy": round(m[f"acc_{f}"], 3),
     "perplexity": round(m[f"ppl_{f}"], 1), "vocab": sizes[f]}
    for f in DYNAMIC_FIELDS
])
print(tbl.to_string(index=False))

## Takeaways

**Ray Train**
- The training loop is plain PyTorch; Ray Train adds the distributed parts — worker launch, dataset sharding, DDP, checkpointing, fault tolerance — without changing the loop.
- `ScalingConfig` is the one knob that moves the same code from one CPU worker here to N GPU workers at `full` (`num_workers`, `use_gpu`); the model fits one GPU at every scale, so it's data-parallel DDP, with `use_fsdp` available if the model ever outgrows a GPU.
- Checkpoints persist to shared storage (`storage_path`), so workers on other nodes can write them and downstream stages can read them.
- The notebook runs the same `train_func` that `scripts/03_pretrain.py` runs headless.

**The model**
- Pretraining is self-supervised masked-feature modeling: predict masked dynamic-field tokens from bidirectional context — no labels required.
- One prediction head per dynamic field, balanced by learned uncertainty weighting so the ~2,000-way merchant head doesn't swamp the 10-way day-of-week head.
- Read per-field perplexity against vocab size to confirm it's learning; the trained encoder becomes the customer embedding in Part 5.

---

## Next

**Part 5 — Batch embedding extraction**: run the pretrained encoder over every card with Ray Data — a heterogeneous CPU-read + GPU-infer pipeline that writes one embedding per window.